# Part 1. Writing Sample: Geopolitical Shock Transmission in Emerging Bond Markets

## Evidence from Kazakhstan Following the 2022 Russian Invasion of Ukraine

**Author:** Darkhan Mashirapov  
**Last updated:** 2026-08  
**Purpose:** Import all pre-cleaned monthly series from `'/Users/darhanmashirapov/Desktop/MSF NU/Writing Sample MS2027/1. Data'`,
validate each one, and merge into separate files for each group of variables.

*Writing Sample*

---

## 01. Data Sources and Pre-Processing

All raw series were downloaded directly from the primary sources listed in
`data_dictionary` below (central banks, stock exchanges, national statistics
agencies, and FRED). **Cleaning, date-format standardization, and daily→monthly
aggregation were performed in Excel prior to import**, not in this notebook,
for two reasons: (1) several source files required manual inspection of
non-standard date formats (DMY/MDY/YDM depending on provider) before they
could be safely parsed programmatically, and (2) this keeps the analysis code
focused on estimation rather than data janitorial work.

This notebook begins at the point where every series has already been:
- converted to a common monthly key format (`YYYY-MM-01`)
- aggregated to monthly frequency using the rule appropriate to its type
  (see `agg_rule` column in the data dictionary)
- exported as a standalone `.xlsx` file, one per variable

The Excel-stage transformations are summarized in the data dictionary table
below for full traceability; the underlying `.xlsx` files are included in
`/data/raw_cleaned/` for anyone who wants to audit a specific series.

**Aggregation rules by variable type:**
| Type | Rule |
|---|---|
| Yields / interbank rates | Monthly average of daily values |
| FX rates, stock/index levels | Monthly average of daily values |
| NSS/ZCYC sovereign curve params | Published monthly parameters (KASE, MOEX) dayly — Monthly averages were calculated |
| Quarterly stock/flow variables (GDP, external debt) | Disaggregated to monthly — see method below |
| Already-monthly series | Used as published |

**Quarterly → monthly disaggregation:**
- **GDP**: proportional (pro-rata) distribution using each month's share of
  quarterly industrial production, following the standard related-indicator
  approach to temporal disaggregation (cf. Denton/Chow-Lin). No smoothing
  adjustment was applied at quarter boundaries.
- **External debt**: linear interpolation between quarterly endpoints, since
  no suitable higher-frequency related indicator was available and the
  series is monotonically trending over the sample.
- **Current account balance**: left at quarterly frequency / excluded from
  monthly-frequency specifications (state whichever you finalize).

**Sample period:** 2019–2026 (monthly). Note that KZT/USD, TONIA, and CPI
exhibit known structural breaks around the 2015 currency float, which fall
outside the estimation window and are therefore not treated further.

In [1]:
import pandas as pd

data_dictionary = pd.DataFrame([
    # --- Dependent variables ---
    dict(var_id=1, var_name="kz_corp_yield_m", name="Corporate bond yield index (KZ)", country="KZ",
         source="KASE", url="https://kase.kz/ru/indexes-and-indicators/bonds/kase-bmy",
         raw_freq="Daily", agg_rule="Monthly average", excel_step="Average, DMY parse"),
    dict(var_id=2, var_name="ru_corp_yield_m", name="Corporate bond yield index (RU)", country="RU",
         source="MOEX", url="https://moex.com/en/index/RUCBITR/archive",
         raw_freq="Daily", agg_rule="Monthly average", excel_step="Average, DMY parse"),
    dict(var_id="3-5", var_name="kz_govt_yield_1y_m / kz_govt_yield_5y_m / kz_govt_yield_10y_m",
         name="Govt bond yield 1/5/10yr (KZ)", country="KZ",
         source="KASE (published NSS/yield curve parameters)",
         url="https://kase.kz/en/markets/markets-valuation/yield-curve-parameters",
         raw_freq="Monthly (published)", agg_rule="As published", excel_step="None — sourced directly"),
    dict(var_id="6-8", var_name="ru_govt_yield_1y_m / ru_govt_yield_5y_m / ru_govt_yield_10y_m",
         name="Govt bond yield 1/5/10yr (RU)", country="RU",
         source="MOEX (published ZCYC parameters)",
         url="https://www.moex.com/download/zcyc/zcyc_parameters_archive.csv.zip",
         raw_freq="Daily (published curve params)", agg_rule="Monthly average", excel_step="Average, DMY parse"),

    # --- KZ fundamentals ---
    dict(var_id=9, var_name="kz_tonia_m", name="TONIA (interbank rate)", country="KZ", source="KASE",
         url="https://kase.kz/en/indexes-and-indicators/repo/tonia",
         raw_freq="Daily", agg_rule="Monthly average", excel_step="MDY parse; 2015 float-regime outliers noted, outside sample window"),
    dict(var_id=10, var_name="kz_usdfx_m", name="KZT/USD exchange rate", country="KZ", source="National Bank of KZ",
         url="https://nationalbank.kz/en/exchangerates/ezhednevnye-oficialnye-rynochnye-kursy-valyut",
         raw_freq="Daily", agg_rule="Monthly average", excel_step="YDM parse; 2015 outliers, outside sample window"),
    dict(var_id=11, var_name="kz_inflation_yoy_m", name="Inflation rate (index)", country="KZ", source="Bureau of National Statistics (taldau.stat.gov.kz)",
         url="https://taldau.stat.gov.kz/en/PivotGrid/PivotTable?indicators=703076&periodId=5&dics=67,848,2753",
         raw_freq="Monthly", agg_rule="As published", excel_step="YDM parse"),
    dict(var_id=12, var_name="kz_gdp_m", name="GDP", country="KZ", source="taldau.stat.gov.kz",
         url="https://taldau.stat.gov.kz/en/", raw_freq="Quarterly",
         agg_rule="Monthly (pro-rata via industrial production shares)",
         excel_step="Proportional disaggregation, see markdown above"),
    dict(var_id=13, var_name="kz_extdebt_m", name="External debt (USD)", country="KZ", source="National Bank of KZ",
         url="https://data.nationalbank.kz/statistics", raw_freq="Quarterly",
         agg_rule="Monthly (linear interpolation)", excel_step="Linear interpolation between quarter endpoints"),
    dict(var_id=14, var_name="kz_reserves_m", name="International reserves (USD)", country="KZ", source="National Bank of KZ",
         url="https://nationalbank.kz/en/international-reserve-and-asset/mezhdunarodnye-rezervy-i-aktivy-nacionalnogo-fonda-rk",
         raw_freq="Monthly", agg_rule="As published", excel_step="Format standardization only"),
    dict(var_id=15, var_name="kz_m2_m", name="M2 money supply (mln tenge)", country="KZ", source="National Bank of KZ",
         url="https://nationalbank.kz/en/...", raw_freq="Monthly", agg_rule="As published",
         excel_step="Format standardization only"),
    dict(var_id=16, var_name="kz_indprod_m", name="Industrial production index", country="KZ", source="taldau.stat.gov.kz",
         url="https://taldau.stat.gov.kz/en/PivotGrid/PivotTable?indicators=701625&periodId=7&dics=68,4303,848",
         raw_freq="Monthly", agg_rule="As published", excel_step="None"),
    dict(var_id=17, var_name="kz_unemp_m", name="Unemployment rate", country="KZ", source="taldau.stat.gov.kz",
         url="https://taldau.stat.gov.kz/en/PivotGrid/PivotTable?indicators=702944&periodId=7&dics=67,749,576,1773,1793",
         raw_freq="Monthly", agg_rule="As published", excel_step="None"),
    dict(var_id=18, var_name="kz_stockidx_m", name="KASE stock index", country="KZ", source="KASE",
         url="https://kase.kz/en/indexes-and-indicators/shares/kase-index",
         raw_freq="Daily", agg_rule="Monthly average (level; NOT a volatility measure — relabel or add realized vol)",
         excel_step="Average"),

    # --- RU fundamentals ---
    dict(var_id=19, var_name="ru_ruonia_m", name="RUONIA (interbank rate)", country="RU", source="CBR",
         url="https://www.cbr.ru/eng/hd_base/ruonia/dynamics/", raw_freq="Daily",
         agg_rule="Monthly average", excel_step="Average"),
    dict(var_id=20, var_name="ru_usdfx_m", name="RUB/USD exchange rate", country="RU", source="CBR",
         url="https://www.cbr.ru/eng/currency_base/dynamics/", raw_freq="Daily",
         agg_rule="Monthly average", excel_step="Average"),
    dict(var_id=21, var_name="ru_inflation_yoy_m", name="Year on year inflation rate", country="RU", source="rateinflation.com",
         url="https://www.rateinflation.com/inflation-rate/russia-historical-inflation-rate/",
         raw_freq="Monthly", agg_rule="As published", excel_step="None"),
    dict(var_id=22, var_name="ru_gdp_m", name="GDP", country="RU", source="FRED (NGDPRNSAXDCRUQ)",
         url="https://fred.stlouisfed.org/series/NGDPRNSAXDCRUQ", raw_freq="Quarterly",
         agg_rule="Monthly (pro-rata via industrial production shares)",
         excel_step="Proportional disaggregation, same method as KZ GDP"),
    dict(var_id=23, var_name="ru_extdebt_m", name="External debt (USD)", country="RU", source="CBR",
         url="https://www.cbr.ru/eng/statistics/macro_itm/external_sector/pb/p_balance/",
         raw_freq="Monthly", agg_rule="As published", excel_step="Format standardization"),
    dict(var_id=24, var_name="ru_reserves_m", name="International reserves (USD)", country="RU", source="CBR",
         url="https://www.cbr.ru/eng/hd_base/mrrf/mrrf_7d/", raw_freq="Daily",
         agg_rule="Monthly average", excel_step="Average"),
    dict(var_id=25, var_name="ru_m2_m", name="M2 money supply", country="RU", source="CBR",
         url="https://www.cbr.ru/eng/statistics/macro_itm/dkfs/monetary_agg/",
         raw_freq="Monthly", agg_rule="As published", excel_step="Format standardization"),
    dict(var_id=26, var_name="ru_indprod_m", name="Industrial production index", country="RU", source="Rosstat",
         url="https://eng.rosstat.gov.ru/folder/160366", raw_freq="Monthly",
         agg_rule="As published", excel_step="None"),
    dict(var_id=27, var_name="ru_unemp_m", name="Unemployment rate", country="RU", source="Rosstat",
         url="https://eng.rosstat.gov.ru/folder/160366", raw_freq="Monthly",
         agg_rule="As published", excel_step="None"),
    dict(var_id=28, var_name="ru_stockidx_m", name="MOEX stock index", country="RU", source="MOEX",
         url="https://www.moex.com/en", raw_freq="Daily",
         agg_rule="Monthly average (level; NOT a volatility measure — relabel or add realized vol)",
         excel_step="Average"),

    # --- Global / common ---
    dict(var_id=29, var_name="gl_oil_brent_m", name="Brent crude oil price", country="Global", source="FRED",
         url="https://fred.stlouisfed.org/series/DCOILBRENTEU", raw_freq="Daily",
         agg_rule="Monthly average", excel_step="Average"),
    dict(var_id=30, var_name="gl_gold_m", name="Gold price", country="Global", source="investing.com",
         url="https://www.investing.com/commodities/gold-historical-data", raw_freq="Daily",
         agg_rule="Monthly average", excel_step="Average"),
    dict(var_id=31, var_name="us_yield_1y_m", name="US Treasury yield 1yr", country="Global", source="FRED",
         url="https://fred.stlouisfed.org/series/DGS1", raw_freq="Daily",
         agg_rule="Monthly average", excel_step="Average"),
    dict(var_id=32, var_name="us_yield_5y_m", name="US Treasury yield 5yr", country="Global", source="FRED",
         url="https://fred.stlouisfed.org/series/DGS5", raw_freq="Daily",
         agg_rule="Monthly average", excel_step="Average"),
    dict(var_id=33, var_name="us_yield_10y_m", name="US Treasury yield 10yr", country="Global", source="FRED",
         url="https://fred.stlouisfed.org/series/DGS10", raw_freq="Daily",
         agg_rule="Monthly average", excel_step="Average"),
    dict(var_id=34, var_name="us_fedfunds_m", name="US Federal Funds Rate", country="Global", source="FRED",
         url="https://fred.stlouisfed.org/series/FEDFUNDS", raw_freq="Monthly",
         agg_rule="As published", excel_step="None"),
    dict(var_id=35, var_name="gl_vix_m", name="VIX", country="Global", source="CBOE",
         url="", raw_freq="Daily", agg_rule="Monthly average", excel_step="Average"),
])

data_dictionary

,var_id,var_name,name,country,source,url,raw_freq,agg_rule,excel_step
0,1,kz_corp_yield_m,Corporate bond yield index (KZ),KZ,KASE,https://kase.kz/ru/indexes-and-indicators/bond...,Daily,Monthly average,"Average, DMY parse"
1,2,ru_corp_yield_m,Corporate bond yield index (RU),RU,MOEX,https://moex.com/en/index/RUCBITR/archive,Daily,Monthly average,"Average, DMY parse"
2,3-5,kz_govt_yield_1y_m / kz_govt_yield_5y_m / kz_g...,Govt bond yield 1/5/10yr (KZ),KZ,KASE (published NSS/yield curve parameters),https://kase.kz/en/markets/markets-valuation/y...,Monthly (published),As published,None — sourced directly
3,6-8,ru_govt_yield_1y_m / ru_govt_yield_5y_m / ru_g...,Govt bond yield 1/5/10yr (RU),RU,MOEX (published ZCYC parameters),https://www.moex.com/download/zcyc/zcyc_parame...,Daily (published curve params),Monthly average,"Average, DMY parse"
4,9,kz_tonia_m,TONIA (interbank rate),KZ,KASE,https://kase.kz/en/indexes-and-indicators/repo...,Daily,Monthly average,"MDY parse; 2015 float-regime outliers noted, o..."
5,10,kz_usdfx_m,KZT/USD exchange rate,KZ,National Bank of KZ,https://nationalbank.kz/en/exchangerates/ezhed...,Daily,Monthly average,"YDM parse; 2015 outliers, outside sample window"
6,11,kz_inflation_yoy_m,Inflation rate (index),KZ,Bureau of National Statistics (taldau.stat.gov...,https://taldau.stat.gov.kz/en/PivotGrid/PivotT...,Monthly,As published,YDM parse
7,12,kz_gdp_m,GDP,KZ,taldau.stat.gov.kz,https://taldau.stat.gov.kz/en/,Quarterly,Monthly (pro-rata via industrial production sh...,"Proportional disaggregation, see markdown above"
8,13,kz_extdebt_m,External debt (USD),KZ,National Bank of KZ,https://data.nationalbank.kz/statistics,Quarterly,Monthly (linear interpolation),Linear interpolation between quarter endpoints
9,14,kz_reserves_m,International reserves (USD),KZ,National Bank of KZ,https://nationalbank.kz/en/international-reser...,Monthly,As published,Format standardization only


## 02. Series Import

Each series is loaded from its own `.xlsx` file (`Final Cleaned` sheet),
date-parsed, renamed to its canonical `var` name, and stored in a
dictionary `raw_series` keyed by variable name.
A validation report is printed for every series: date range, observation
count, missing values, and min/max — so any import problem is immediately
visible.

In [2]:
# ── Standard library ────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

# ── Core ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator
import matplotlib.gridspec as gridspec

# ── Display settings ────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_rows", 60)

print("✅ Libraries loaded successfully")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")

✅ Libraries loaded successfully
   pandas  2.3.3
   numpy   2.3.5


In [3]:
# ── Set base path — update this if you move the project folder ──────────────
BASE = Path("/Users/darhanmashirapov/Desktop/MSF NU/Writing Sample MS2027/1. Data")

# Confirm it exists
assert BASE.exists(), f"❌ BASE path not found: {BASE}"
print(f"✅ Project root: {BASE}")

✅ Project root: /Users/darhanmashirapov/Desktop/MSF NU/Writing Sample MS2027/1. Data


## 03. Dependent Variables block

In [4]:
# =============================================================================
# BLOCK 1 · DEPENDENT VARIABLES
# =============================================================================

import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ── File paths ────────────────────────────────────────────────────────────────
BASE = "/Users/darhanmashirapov/Desktop/MSF NU/Writing Sample MS2027/1. Data/1-8. Dependent Variables"

files = {
    "kz_corp_yield_m"    : f"{BASE}/Kazakhstan/1. Corporate bond yield index (Kazakhstan)/Corporate bond yield index (Kazakhstan).xlsx",
    "ru_corp_yield_m" : f"{BASE}/Russia/2. Corporate Bond Yield Index /Corporate bond yield index (Russia).xlsx",
    "kz_govt_yields"     : f"{BASE}/Kazakhstan/3. 4. 5 Government Bond Yield Data/Kazakhstan Government Yield Data.xlsx",
    "ru_govt_yields"     : f"{BASE}/Russia/6. 7. 8 Government Bond Yield Data/Russian Givernment Bond Yield Data.xlsx",
}

# ── Load ──────────────────────────────────────────────────────────────────────
kz_corp  = pd.read_excel(files["kz_corp_yield_m"],  sheet_name="Final Cleaned", parse_dates=["date"])
ru_corp  = pd.read_excel(files["ru_corp_yield_m"],  sheet_name="Final Cleaned", parse_dates=["date"])
kz_govt  = pd.read_excel(files["kz_govt_yields"],   sheet_name="Final Cleaned", parse_dates=["date"])
ru_govt  = pd.read_excel(files["ru_govt_yields"],   sheet_name="Final Cleaned", parse_dates=["date"])

# ── Display ───────────────────────────────────────────────────────────────────
for name, df in [("KZ Corporate Yield", kz_corp),
                 ("RU Corporate Yield", ru_corp),
                 ("KZ Government Yields (1/5/10yr)", kz_govt),
                 ("RU Government Yields (1/5/10yr)", ru_govt)]:
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"  Rows: {len(df)}  |  Columns: {df.columns.tolist()}")
    print(f"{'='*55}")
    display(df.head(3))
    display(df.tail(3))


  KZ Corporate Yield
  Rows: 314  |  Columns: ['date', 'kz_corp_yield_m']


,date,kz_corp_yield_m
0,2000-07-01,12.3079
1,2000-08-01,12.1359
2,2000-09-01,12.0795


,date,kz_corp_yield_m
311,2026-06-01,16.5986
312,2026-07-01,16.4673
313,2026-08-01,16.4300



  RU Corporate Yield
  Rows: 164  |  Columns: ['date', 'ru_corp_yield_m']


,date,ru_corp_yield_m
0,2013-01-01,8.4594
1,2013-02-01,8.0670
2,2013-03-01,8.0760


...

,date,kz_govt_yield_1y_m,kz_govt_yield_5y_m,kz_govt_yield_10y_m
0,2026-08-01,0.1469,0.1357,0.1280
1,2026-07-01,0.1489,0.1358,0.1302
2,2026-06-01,0.1646,0.1446,0.1355


,date,kz_govt_yield_1y_m,kz_govt_yield_5y_m,kz_govt_yield_10y_m
79,2020-01-01,0.0958,0.0910,0.0902
80,2019-12-01,0.0960,0.0906,0.0894
81,2019-11-01,0.0955,0.0897,0.0883



  RU Government Yields (1/5/10yr)
  Rows: 284  |  Columns: ['date', 'ru_govt_yield_1y_m', 'ru_govt_yield_5y_m', 'ru_govt_yield_10y_m']


,date,ru_govt_yield_1y_m,ru_govt_yield_5y_m,ru_govt_yield_10y_m
0,2026-08-01,13.6740,15.4320,15.7040
1,2026-07-01,13.9309,15.7491,16.2857
2,2026-06-01,13.0914,14.7290,15.3862


,date,ru_govt_yield_1y_m,ru_govt_yield_5y_m,ru_govt_yield_10y_m
281,2003-03-01,7.9175,9.6480,9.6565
282,2003-02-01,9.4142,11.0811,11.0821
283,2003-01-01,12.0045,13.3380,13.2995


In [5]:
# =============================================================================
# BLOCK 2 · MERGE + UNIT STANDARDIZATION
# =============================================================================

# ── Fix KZ government yields: decimal fraction -> percentage points ──────────
# (0.1469 -> 14.6900), so all four dataframes are on the same scale
kz_govt_cols = ["kz_govt_yield_1y_m", "kz_govt_yield_5y_m", "kz_govt_yield_10y_m"]
kz_govt[kz_govt_cols] = kz_govt[kz_govt_cols] * 100

# ── Merge on date, outer join to preserve full history across mismatched
# start dates (KZ corp from 2000, RU corp from 2013, KZ govt from 2019,
# RU govt from 2003) ──────────────────────────────────────────────────────────
merged = (
    kz_corp
    .merge(ru_corp, on="date", how="outer")
    .merge(kz_govt, on="date", how="outer")
    .merge(ru_govt, on="date", how="outer")
    .sort_values("date")
    .reset_index(drop=True)
)

# ── Sanity check: confirm all columns now sit in comparable percentage-point
# range (rough median check, not a formal test) ───────────────────────────────
print("Median values by column (all should be in similar percentage-point range):")
print(merged.drop(columns="date").median().round(4))

print(f"\nMerged shape: {merged.shape}")
print(f"Date range: {merged['date'].min().date()} to {merged['date'].max().date()}")
print(f"\nMissing values per column:\n{merged.isna().sum()}")

display(merged.head(3))
display(merged.tail(3))

Median values by column (all should be in similar percentage-point range):
kz_corp_yield_m       10.8390
ru_corp_yield_m        9.7561
kz_govt_yield_1y_m    12.5666
kz_govt_yield_5y_m    11.4344
kz_govt_yield_10y_m   10.5310
ru_govt_yield_1y_m     6.8193
ru_govt_yield_5y_m     7.8475
ru_govt_yield_10y_m    8.3579
dtype: float64

Merged shape: (314, 9)
Date range: 2000-07-01 to 2026-08-01

Missing values per column:
date                     0
kz_corp_yield_m          0
ru_corp_yield_m        150
kz_govt_yield_1y_m     232
kz_govt_yield_5y_m     232
kz_govt_yield_10y_m    232
ru_govt_yield_1y_m      30
ru_govt_yield_5y_m      30
ru_govt_yield_10y_m     30
dtype: int64


,date,kz_corp_yield_m,ru_corp_yield_m,kz_govt_yield_1y_m,kz_govt_yield_5y_m,kz_govt_yield_10y_m,ru_govt_yield_1y_m,ru_govt_yield_5y_m,ru_govt_yield_10y_m
0,2000-07-01,12.3079,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-08-01,12.1359,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-09-01,12.0795,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,date,kz_corp_yield_m,ru_corp_yield_m,kz_govt_yield_1y_m,kz_govt_yield_5y_m,kz_govt_yield_10y_m,ru_govt_yield_1y_m,ru_govt_yield_5y_m,ru_govt_yield_10y_m
311,2026-06-01,16.5986,15.2267,16.4597,14.4632,13.5528,13.0914,14.7290,15.3862
312,2026-07-01,16.4673,16.2513,14.8907,13.5775,13.0210,13.9309,15.7491,16.2857
313,2026-08-01,16.4300,15.7480,14.6852,13.5695,12.8018,13.6740,15.4320,15.7040


In [6]:
# =============================================================================
# BLOCK 3 · SAVE MERGED DEPENDENT VARIABLES
# =============================================================================

output_path = f"{BASE}/dependent_variables_merged.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    merged.to_excel(writer, sheet_name="Final Cleaned", index=False)

print(f"Saved: {output_path}")
print(f"Shape: {merged.shape}")

Saved: /Users/darhanmashirapov/Desktop/MSF NU/Writing Sample MS2027/1. Data/1-8. Dependent Variables/dependent_variables_merged.xlsx
Shape: (314, 9)


## 04. Domestic Fundamentals Kazakhstan

In [7]:
!pip install xlrd

In [8]:
# =============================================================================
# BLOCK 4 · KZ DOMESTIC FUNDAMENTALS (vars 9–18)
# =============================================================================
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

BASE_KZ = "/Users/darhanmashirapov/Desktop/MSF NU/Writing Sample MS2027/1. Data/9-18. Domestic Fundamentals Kazakhstan"

files_kz = {
    "kz_tonia_m"        : (f"{BASE_KZ}/9. TONIA (interbank rate)/TONIA_260810_Kazakhstan.xlsx", "Final Cleaned"),
    "kz_usdfx_m"        : (f"{BASE_KZ}/10. Exchange Rate USD:KZT/Официальные (рыночные) курсы валют.xlsx", "Final Cleaned"),
    "kz_inflation_yoy_m": (f"{BASE_KZ}/11. CPI Kazakhstan /CPI_Kazakhstan_reporting period to the previous period.xlsx", "Final Cleaned"),
    "kz_gdp_q"          : (f"{BASE_KZ}/12. GDP Quarterly/GDP Quarterly Данные 2026.08.10-23.48.56..xls", "Final Cleaned"),
    "kz_extdebt_q"      : (f"{BASE_KZ}/13. External Debt/External debt Kazakhstan.xlsx", "Final Cleaned"),
    "kz_reserves_m"     : (f"{BASE_KZ}/14. International Reserves/International Reserves and foreign currency assets of the National Fund of Republic of Kazakhstan.xlsx", "Final Cleaned"),
    "kz_m2_m"           : (f"{BASE_KZ}/15. M2 money supply/M2 money supply.xlsx", "Final Cleaned"),
    "kz_indprod_m"      : (f"{BASE_KZ}/16. Industrial production index/Industrial production index Kazakhstan.xlsx", "Final Cleaned"),
    "kz_unemp_m"        : (f"{BASE_KZ}/17. Unemployment Rate/Unemployment Rate Kazakhstan.xlsx", "Final Cleaned"),
    "kz_stockidx_m"     : (f"{BASE_KZ}/18. Kase Index/Index_KASE_260811.xlsx", "Final Cleaned"),
}


def smart_read(path, sheet_name):
    try:
        return pd.read_excel(
            path,
            sheet_name=sheet_name,
            parse_dates=["date"]
        )
    except Exception:
        try:
            return pd.read_excel(
                path,
                sheet_name=sheet_name,
                engine="xlrd",
                parse_dates=["date"]
            )
        except Exception:
            df = pd.read_html(path)[0]
            df.columns = ["date", df.columns[-1]]
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
            return df


kz_data = {}

for var_name, (path, sheet) in files_kz.items():

    try:
        df = smart_read(path, sheet)
        kz_data[var_name] = df

        print(f"\n{'='*60}\n  {var_name}\n  Rows: {len(df)}  |  Columns: {df.columns.tolist()}\n{'='*60}")
        display(df.head(3))
        display(df.tail(3))

    except Exception as e:
        print(f"\n❌ {var_name} failed: {e}")


  kz_tonia_m
  Rows: 300  |  Columns: ['date', 'kz_tonia_m']


,date,kz_tonia_m
0,2001-09-01,3.3929
1,2001-10-01,3.5843
2,2001-11-01,5.4329


,date,kz_tonia_m
297,2026-06-01,17.0491
298,2026-07-01,16.5264
299,2026-08-01,15.7800



  kz_usdfx_m
  Rows: 313  |  Columns: ['date', 'kz_usdfx_m']


,date,kz_usdfx_m
0,2000-08-01,142.6583
1,2000-09-01,142.7167
2,2000-10-01,142.6419


...

,date,kz_unemp_m
0,2013-04-01,5.3000
1,2013-05-01,5.3000
2,2013-06-01,5.2000


,date,kz_unemp_m
154,2026-02-01,4.5000
155,2026-03-01,4.5000
156,2026-04-01,4.5000



  kz_stockidx_m
  Rows: 314  |  Columns: ['date', 'kz_stockidx_m']


,date,kz_stockidx_m
0,2000-07-01,101.5829
1,2000-08-01,103.9541
2,2000-09-01,110.2948


,date,kz_stockidx_m
311,2026-06-01,"7,731.8836"
312,2026-07-01,"7,700.9800"
313,2026-08-01,"7,881.1650"


In [9]:
# =============================================================================
# BLOCK 5 · CLEAN, MERGE & SAVE KZ DOMESTIC FUNDAMENTALS (vars 9–18)
# =============================================================================
import numpy as np

# ── Fix column name typo ──────────────────────────────────────────────────────
kz_data["kz_inflation_yoy_m"] = kz_data["kz_inflation_yoy_m"].rename(
    columns={"kz_iflation_yoy_m": "kz_inflation_yoy_m"}
)

# ── Clean comma-formatted numeric strings → float ────────────────────────────
def clean_numeric(df, col):
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(",", "", regex=False).astype(float)
    return df

kz_data["kz_gdp_q"]       = clean_numeric(kz_data["kz_gdp_q"], "kz_gdp_m")
kz_data["kz_extdebt_q"]   = clean_numeric(kz_data["kz_extdebt_q"], "kz_extdebt_m")
kz_data["kz_stockidx_m"]  = clean_numeric(kz_data["kz_stockidx_m"], "kz_stockidx_m")

# ── GDP: replace trailing placeholder zeros (unpublished periods) with NaN ──
gdp_df = kz_data["kz_gdp_q"]
gdp_df.loc[gdp_df["kz_gdp_m"] == 0, "kz_gdp_m"] = np.nan
kz_data["kz_gdp_q"] = gdp_df

# ── Inflation: decimal fraction → percentage points (0.0743 -> 7.43) ─────────
kz_data["kz_inflation_yoy_m"]["kz_inflation_yoy_m"] *= 100

# ── External debt: genuinely quarterly — reindex to monthly + linear interp ──
extdebt_df = kz_data["kz_extdebt_q"].sort_values("date").set_index("date")
extdebt_monthly = extdebt_df.resample("MS").asfreq()          # insert monthly rows
extdebt_monthly["kz_extdebt_m"] = extdebt_monthly["kz_extdebt_m"].interpolate(method="linear")
extdebt_monthly = extdebt_monthly.reset_index()
kz_data["kz_extdebt_q"] = extdebt_monthly

# ── Merge all 10 series on date, outer join to preserve full history ────────
dfs = list(kz_data.values())
merged_kz = dfs[0]
for df in dfs[1:]:
    merged_kz = merged_kz.merge(df, on="date", how="outer")

merged_kz = merged_kz.sort_values("date").reset_index(drop=True)

# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f"Merged shape: {merged_kz.shape}")
print(f"Date range: {merged_kz['date'].min().date()} to {merged_kz['date'].max().date()}")
print(f"\nMissing values per column:\n{merged_kz.isna().sum()}")
print(f"\nDtypes:\n{merged_kz.dtypes}")

display(merged_kz.head(3))
display(merged_kz.tail(3))

# ── Save ──────────────────────────────────────────────────────────────────────
output_path_kz = f"{BASE_KZ}/kz_domestic_fundamentals_merged.xlsx"
with pd.ExcelWriter(output_path_kz, engine="openpyxl") as writer:
    merged_kz.to_excel(writer, sheet_name="Final Cleaned", index=False)

print(f"\nSaved: {output_path_kz}")

Merged shape: (385, 11)
Date range: 1994-08-01 to 2026-08-01

Missing values per column:
date                    0
kz_tonia_m             85
kz_usdfx_m             72
kz_inflation_yoy_m    210
kz_gdp_m              238
kz_extdebt_m           81
kz_reserves_m           1
kz_m2_m                74
kz_indprod_m          235
kz_unemp_m            228
kz_stockidx_m          71
dtype: int64

Dtypes:
date                  datetime64[ns]
kz_tonia_m                   float64
kz_usdfx_m                   float64
kz_inflation_yoy_m           float64
kz_gdp_m                     float64
kz_extdebt_m                 float64
kz_reserves_m                float64
kz_m2_m                      float64
kz_indprod_m                 float64
kz_unemp_m                   float64
kz_stockidx_m                float64
dtype: object


,date,kz_tonia_m,kz_usdfx_m,kz_inflation_yoy_m,kz_gdp_m,kz_extdebt_m,kz_reserves_m,kz_m2_m,kz_indprod_m,kz_unemp_m,kz_stockidx_m
0,1994-08-01,NaN,NaN,NaN,NaN,NaN,"1,099.0000",NaN,NaN,NaN,NaN
1,1994-09-01,NaN,NaN,NaN,NaN,NaN,"1,118.0000",NaN,NaN,NaN,NaN
2,1994-10-01,NaN,NaN,NaN,NaN,NaN,968.0000,NaN,NaN,NaN,NaN


,date,kz_tonia_m,kz_usdfx_m,kz_inflation_yoy_m,kz_gdp_m,kz_extdebt_m,kz_reserves_m,kz_m2_m,kz_indprod_m,kz_unemp_m,kz_stockidx_m
382,2026-06-01,17.0491,487.8660,10.4691,NaN,NaN,"61,829.0000","48,128,861.0000",112.9000,NaN,"7,731.8836"
383,2026-07-01,16.5264,471.7445,10.4691,NaN,NaN,"63,715.0000",NaN,NaN,NaN,"7,700.9800"
384,2026-08-01,15.7800,471.5250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"7,881.1650"



Saved: /Users/darhanmashirapov/Desktop/MSF NU/Writing Sample MS2027/1. Data/9-18. Domestic Fundamentals Kazakhstan/kz_domestic_fundamentals_merged.xlsx


## 05. Domestic Fundamentals Russia

In [10]:
# =============================================================================
# BLOCK 6 · RU DOMESTIC FUNDAMENTALS (vars 19–28) — LOAD & INSPECT
# =============================================================================
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

BASE_RU = "/Users/darhanmashirapov/Desktop/MSF NU/Writing Sample MS2027/1. Data/19-28. Domestic Fundamentals Russia"

files_ru = {
    "ru_ruonia_m"        : (f"{BASE_RU}/19. RUONIA /RUONIA (interbank rate)- Russia.xlsx", "Final Cleaned"),
    "ru_usdfx_m"         : (f"{BASE_RU}/20. EXCHANGE RATE USD:RUB/RUB:USD exchange rate.xlsx", "Final Cleaned"),
    "ru_inflation_yoy_m" : (f"{BASE_RU}/21. Inflation CPI Russia/Inflation YoY Russia_final.xlsx", "Final Cleaned"),
    "ru_gdp_m"           : (f"{BASE_RU}/22. GDP Russia/GDP (quarterly) Russian in Ruble.xlsx", "Final Cleaned"),
    "ru_extdebt_m"       : (f"{BASE_RU}/23. External Debt Russia/External Debt of Russia.xlsx", "Final Cleaned"),
    "ru_reserves_m"      : (f"{BASE_RU}/24. International Reserves/International Reserves Russia_V2.xlsx", "Final Cleaned"),
    "ru_m2_m"            : (f"{BASE_RU}/25. M2 Money Supply Russia/monetary_agg_e.xlsx", "Final Cleaned"),
    "ru_indprod_m"       : (f"{BASE_RU}/26. Industrial Production Index/SDDS_industrial production index_2026.xlsx", "Final Cleaned"),
    "ru_unemp_m"         : (f"{BASE_RU}/27. Unemployment Rate/Unemployment Rate Russia.xlsx", "Final Cleaned"),
    "ru_stockidx_m"      : (f"{BASE_RU}/28. MOEX Index/MOEX Index in RUB.xlsx", "Final Cleaned"),
}

# smart_read must already be defined from Block 4 — re-run its definition
# below if this kernel session doesn't have it.
def smart_read(path, sheet_name=None):
    with open(path, "rb") as f:
        header = f.read(8)

    if header.startswith(b"PK"):
        return pd.read_excel(path, sheet_name=sheet_name, engine="openpyxl")
    elif header.startswith(b"\xd0\xcf\x11\xe0"):
        return pd.read_excel(path, sheet_name=sheet_name, engine="xlrd")
    elif header.startswith(b"<html") or header.startswith(b"<!DOC") or header.startswith(b"<?xml"):
        last_err = None
        for encoding in ["utf-8", "cp1251", "windows-1251", "latin1"]:
            for flavor in ["lxml", "html5lib", "bs4"]:
                try:
                    tables = pd.read_html(path, encoding=encoding, flavor=flavor)
                    print(f"  ✓ Parsed using encoding={encoding}, flavor={flavor}")
                    if len(tables) > 1:
                        print(f"  ⚠ {len(tables)} tables found — using table[0], verify")
                    return tables[0]
                except Exception as e:
                    last_err = e
                    continue
        raise ValueError(f"Could not parse HTML file {path}. Last error: {last_err}")
    else:
        raise ValueError(f"Unrecognized file signature {header!r} for {path}")


ru_data = {}
for var_name, (path, sheet) in files_ru.items():
    df = smart_read(path, sheet_name=sheet)
    ru_data[var_name] = df
    print(f"\n{'='*60}\n  {var_name}\n  Rows: {len(df)}  |  Columns: {df.columns.tolist()}\n{'='*60}")
    display(df.head(3))
    display(df.tail(3))


  ru_ruonia_m
  Rows: 193  |  Columns: ['date', 'ru_ruonia_m']


,date,ru_ruonia_m
0,2026-08-01,13.7411
1,2026-07-01,14.2826
2,2026-06-01,14.0314


,date,ru_ruonia_m
190,2010-10-01,2.6462
191,2010-09-01,2.5409
192,2010-08-01,2.5400



  ru_usdfx_m
  Rows: 200  |  Columns: ['date', 'ru_usdfx_m']


,date,ru_usdfx_m
0,2026-08-01,81.9540
1,2026-07-01,77.8817
2,2026-06-01,73.5829


...

,date,ru_unemp_m
0,2015-01-01,0.0549
1,2015-02-01,0.0580
2,2015-03-01,0.0591


,date,ru_unemp_m
135,2026-04-01,0.0220
136,2026-05-01,0.0212
137,2026-06-01,0.0223



  ru_stockidx_m
  Rows: 320  |  Columns: ['date', 'ru_stockidx_m']


,date,ru_stockidx_m
0,2000-01-01,202.3450
1,2000-02-01,200.7076
2,2000-03-01,240.6614


,date,ru_stockidx_m
317,2026-06-01,"2,453.7167"
318,2026-07-01,"2,165.9352"
319,2026-08-01,"2,223.8740"


In [11]:
# =============================================================================
# BLOCK 6+7 COMBINED · fresh reload + safe, idempotent transforms + merge
# =============================================================================
# ── STEP 1: reload all RU files fresh from disk (wipes any stale in-memory state) ──
ru_data = {}
for var_name, (path, sheet) in files_ru.items():
    df = smart_read(path, sheet_name=sheet)
    ru_data[var_name] = df

# ── STEP 2: force date dtype ──────────────────────────────────────────────────
for name, df in ru_data.items():
    ru_data[name]["date"] = pd.to_datetime(df["date"], errors="coerce")
    n_bad = ru_data[name]["date"].isna().sum()
    if n_bad > 0:
        print(f"  ⚠ {name}: {n_bad} rows had unparseable dates -> set to NaT, inspect these")

# ── STEP 3: fix trailing-space column name ───────────────────────────────────
ru_data["ru_gdp_m"] = ru_data["ru_gdp_m"].rename(columns=lambda c: c.strip())

# ── STEP 4: clean comma-formatted numeric strings → float ───────────────────
def clean_numeric(df, col):
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(",", "", regex=False).astype(float)
    return df

ru_data["ru_gdp_m"]      = clean_numeric(ru_data["ru_gdp_m"], "ru_gdp_m")
ru_data["ru_extdebt_m"]  = clean_numeric(ru_data["ru_extdebt_m"], "ru_extdebt_m")
ru_data["ru_m2_m"]       = clean_numeric(ru_data["ru_m2_m"], "ru_m2_m")
ru_data["ru_stockidx_m"] = clean_numeric(ru_data["ru_stockidx_m"], "ru_stockidx_m")

# ── STEP 5: decimal fraction → percentage points, IDEMPOTENT (safe to re-run) ──
def to_percentage_points(df, col, threshold=1.0):
    """Only multiply by 100 if values look like a decimal fraction.
    Re-running this on already-converted data does nothing — safe by design."""
    med = df[col].median()
    if med < threshold:
        df[col] = df[col] * 100
        print(f"  {col}: converted (median was {med:.4f})")
    else:
        print(f"  {col}: left as-is (median {med:.4f}, already percentage points)")
    return df

ru_data["ru_inflation_yoy_m"] = to_percentage_points(ru_data["ru_inflation_yoy_m"], "ru_inflation_yoy_m")
ru_data["ru_unemp_m"]         = to_percentage_points(ru_data["ru_unemp_m"], "ru_unemp_m")

# ── STEP 6: merge all 10 series on date, outer join ──────────────────────────
dfs = list(ru_data.values())
merged_ru = dfs[0]
for df in dfs[1:]:
    merged_ru = merged_ru.merge(df, on="date", how="outer")

merged_ru = merged_ru.sort_values("date").reset_index(drop=True)

# ── STEP 7: sanity checks ─────────────────────────────────────────────────────
print(f"\nMerged shape: {merged_ru.shape}")
print(f"date dtype: {merged_ru['date'].dtype}")
print(f"Date range: {merged_ru['date'].min().date()} to {merged_ru['date'].max().date()}")
print(f"\nMedian values (spot-check magnitudes):\n{merged_ru.drop(columns='date').median().round(4)}")
print(f"\nMissing values per column:\n{merged_ru.isna().sum()}")

display(merged_ru.head(3))
display(merged_ru.tail(3))

# ── STEP 8: save ──────────────────────────────────────────────────────────────
output_path_ru = f"{BASE_RU}/ru_domestic_fundamentals_merged.xlsx"
with pd.ExcelWriter(output_path_ru, engine="openpyxl") as writer:
    merged_ru.to_excel(writer, sheet_name="Final Cleaned", index=False)

print(f"\nSaved: {output_path_ru}")

  ru_inflation_yoy_m: converted (median was 0.0920)
  ru_unemp_m: converted (median was 0.0460)

Merged shape: (405, 11)
date dtype: datetime64[ns]
Date range: 1992-12-01 to 2026-08-01

Median values (spot-check magnitudes):
ru_ruonia_m                   7.6153
ru_usdfx_m                   64.2414
ru_inflation_yoy_m            9.2000
ru_gdp_m             10,895,046.5270
ru_extdebt_m            469,778.5726
ru_reserves_m               447.0525
ru_m2_m                  13,944.1500
ru_indprod_m                 92.2500
ru_unemp_m                    4.5953
ru_stockidx_m             1,633.4516
dtype: float64

Missing values per column:
date                    0
ru_ruonia_m           212
ru_usdfx_m            205
ru_inflation_yoy_m     50
ru_gdp_m              270
ru_extdebt_m          125
ru_reserves_m          65
ru_m2_m                 1
ru_indprod_m          267
ru_unemp_m            267
ru_stockidx_m          85
dtype: int64


,date,ru_ruonia_m,ru_usdfx_m,ru_inflation_yoy_m,ru_gdp_m,ru_extdebt_m,ru_reserves_m,ru_m2_m,ru_indprod_m,ru_unemp_m,ru_stockidx_m
0,1992-12-01,NaN,NaN,NaN,NaN,NaN,NaN,6.5000,NaN,NaN,NaN
1,1993-01-01,NaN,NaN,NaN,NaN,NaN,NaN,7.7000,NaN,NaN,NaN
2,1993-02-01,NaN,NaN,NaN,NaN,NaN,NaN,8.4000,NaN,NaN,NaN


,date,ru_ruonia_m,ru_usdfx_m,ru_inflation_yoy_m,ru_gdp_m,ru_extdebt_m,ru_reserves_m,ru_m2_m,ru_indprod_m,ru_unemp_m,ru_stockidx_m
402,2026-06-01,14.0314,73.5829,6.0000,NaN,NaN,734.5000,"134,848.2000",105.5000,2.2255,"2,453.7167"
403,2026-07-01,14.2826,77.8817,NaN,NaN,NaN,723.8800,"135,683.4000",NaN,NaN,"2,165.9352"
404,2026-08-01,13.7411,81.9540,NaN,NaN,NaN,740.0000,NaN,NaN,NaN,"2,223.8740"



Saved: /Users/darhanmashirapov/Desktop/MSF NU/Writing Sample MS2027/1. Data/19-28. Domestic Fundamentals Russia/ru_domestic_fundamentals_merged.xlsx


## 06. Global/Common Factors

In [12]:
# =============================================================================
# BLOCK 8 · GLOBAL / COMMON FACTORS (vars 29–35) — LOAD & INSPECT
# =============================================================================
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

BASE_GL = "/Users/darhanmashirapov/Desktop/MSF NU/Writing Sample MS2027/1. Data/29-35. Global:Common Factors "

files_gl = {
    "gl_oil_brent_m" : (f"{BASE_GL}/29. Oil Price/Brent crude oil price.xlsx", "Final Cleaned"),
    "gl_gold_m"      : (f"{BASE_GL}/30. Gold Price Futures/Gold Futures Historical Data.xlsx", "Final Cleaned"),
    "us_yield_1y_m"  : (f"{BASE_GL}/31. Market Yield on U.S. Treasury Securities at 1-Year Constant Maturity/U.S. Treasury Securities at 1-Year Constant Maturity YIELD.xlsx", "Final Cleaned"),
    "us_yield_5y_m"  : (f"{BASE_GL}/32. Market Yield on U.S. Treasury Securities at 5-Year Constant Maturity/5 year Yield US.xlsx", "Final Cleaned"),
    "us_yield_10y_m" : (f"{BASE_GL}/33. Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity/10 year Yield US.xlsx", "Final Cleaned"),
    "us_fedfunds_m"  : (f"{BASE_GL}/34. Federal Funds Effective Rate/FEDFUNDS_Monthly.xlsx", "Final Cleaned"),
    "gl_vix_m"       : (f"{BASE_GL}/35. CBOE Volatility Index VIX /Volatility Index.xlsx", "Final Cleaned"),
}

# smart_read must already be defined (from Block 4/6) — re-defining here for
# safety in case this is a fresh kernel session.
def smart_read(path, sheet_name=None):
    with open(path, "rb") as f:
        header = f.read(8)
    if header.startswith(b"PK"):
        return pd.read_excel(path, sheet_name=sheet_name, engine="openpyxl")
    elif header.startswith(b"\xd0\xcf\x11\xe0"):
        return pd.read_excel(path, sheet_name=sheet_name, engine="xlrd")
    elif header.startswith(b"<html") or header.startswith(b"<!DOC") or header.startswith(b"<?xml"):
        last_err = None
        for encoding in ["utf-8", "cp1251", "windows-1251", "latin1"]:
            for flavor in ["lxml", "html5lib", "bs4"]:
                try:
                    tables = pd.read_html(path, encoding=encoding, flavor=flavor)
                    print(f"  ✓ Parsed using encoding={encoding}, flavor={flavor}")
                    if len(tables) > 1:
                        print(f"  ⚠ {len(tables)} tables found — using table[0], verify")
                    return tables[0]
                except Exception as e:
                    last_err = e
                    continue
        raise ValueError(f"Could not parse HTML file {path}. Last error: {last_err}")
    else:
        raise ValueError(f"Unrecognized file signature {header!r} for {path}")


gl_data = {}
for var_name, (path, sheet) in files_gl.items():
    df = smart_read(path, sheet_name=sheet)
    gl_data[var_name] = df
    print(f"\n{'='*60}\n  {var_name}\n  Rows: {len(df)}  |  Columns: {df.columns.tolist()}\n{'='*60}")
    display(df.head(3))
    display(df.tail(3))


  gl_oil_brent_m
  Rows: 472  |  Columns: ['date', 'gl_oil_brent_m']


,date,gl_oil_brent_m
0,1987-05-01,18.5800
1,1987-06-01,18.8605
2,1987-07-01,19.8565


,date,gl_oil_brent_m
469,2026-06-01,85.3991
470,2026-07-01,83.7587
471,2026-08-01,89.3271



  gl_gold_m
  Rows: 200  |  Columns: ['date', 'gl_gold_m']


,date,gl_gold_m
0,2026-08-01,"4,442.5200"
1,2026-07-01,"4,088.7304"
2,2026-06-01,"4,253.9864"


...

,date,us_fedfunds_m
0,2000-01-01,5.4500
1,2000-02-01,5.7300
2,2000-03-01,5.8500


,date,us_fedfunds_m
316,2026-05-01,3.6300
317,2026-06-01,3.6300
318,2026-07-01,3.6300



  gl_vix_m
  Rows: 440  |  Columns: ['date', 'gl_vix_m']


,date,gl_vix_m
0,1990-01-01,23.3473
1,1990-02-01,23.2626
2,1990-03-01,20.0623


,date,gl_vix_m
437,2026-06-01,17.9068
438,2026-07-01,17.0909
439,2026-08-01,15.2390


In [13]:
# =============================================================================
# BLOCK 9 · CLEAN, MERGE & SAVE GLOBAL/COMMON FACTORS (vars 29–35)
# =============================================================================

# ── Force date dtype ───────────────────────────────────────────────────────
for name, df in gl_data.items():
    gl_data[name]["date"] = pd.to_datetime(df["date"], errors="coerce")
    n_bad = gl_data[name]["date"].isna().sum()
    if n_bad > 0:
        print(f"  ⚠ {name}: {n_bad} rows had unparseable dates -> set to NaT, inspect these")

# ── Fix trailing-space column name ─────────────────────────────────────────
gl_data["us_yield_10y_m"] = gl_data["us_yield_10y_m"].rename(columns=lambda c: c.strip())

# ── Clean comma-formatted numeric string → float ───────────────────────────
def clean_numeric(df, col):
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(",", "", regex=False).astype(float)
    return df

gl_data["gl_gold_m"] = clean_numeric(gl_data["gl_gold_m"], "gl_gold_m")

# ── Merge all 7 series on date, outer join to preserve full history ───────
dfs = list(gl_data.values())
merged_gl = dfs[0]
for df in dfs[1:]:
    merged_gl = merged_gl.merge(df, on="date", how="outer")

merged_gl = merged_gl.sort_values("date").reset_index(drop=True)

# ── Sanity checks ───────────────────────────────────────────────────────────
print(f"\nMerged shape: {merged_gl.shape}")
print(f"date dtype: {merged_gl['date'].dtype}")
print(f"Date range: {merged_gl['date'].min().date()} to {merged_gl['date'].max().date()}")
print(f"\nMedian values (spot-check magnitudes):\n{merged_gl.drop(columns='date').median().round(4)}")
print(f"\nMissing values per column:\n{merged_gl.isna().sum()}")

display(merged_gl.head(3))
display(merged_gl.tail(3))

# ── Save ─────────────────────────────────────────────────────────────────────
output_path_gl = f"{BASE_GL}/global_common_factors_merged.xlsx"
with pd.ExcelWriter(output_path_gl, engine="openpyxl") as writer:
    merged_gl.to_excel(writer, sheet_name="Final Cleaned", index=False)

print(f"\nSaved: {output_path_gl}")


Merged shape: (472, 8)
date dtype: datetime64[ns]
Date range: 1987-05-01 to 2026-08-01

Median values (spot-check magnitudes):
gl_oil_brent_m      46.5277
gl_gold_m        1,569.8080
us_yield_1y_m        1.6752
us_yield_5y_m        2.7766
us_yield_10y_m       3.5229
us_fedfunds_m        1.3400
gl_vix_m            17.6733
dtype: float64

Missing values per column:
date                0
gl_oil_brent_m      0
gl_gold_m         272
us_yield_1y_m     152
us_yield_5y_m     152
us_yield_10y_m    152
us_fedfunds_m     153
gl_vix_m           32
dtype: int64


,date,gl_oil_brent_m,gl_gold_m,us_yield_1y_m,us_yield_5y_m,us_yield_10y_m,us_fedfunds_m,gl_vix_m
0,1987-05-01,18.5800,NaN,NaN,NaN,NaN,NaN,NaN
1,1987-06-01,18.8605,NaN,NaN,NaN,NaN,NaN,NaN
2,1987-07-01,19.8565,NaN,NaN,NaN,NaN,NaN,NaN


,date,gl_oil_brent_m,gl_gold_m,us_yield_1y_m,us_yield_5y_m,us_yield_10y_m,us_fedfunds_m,gl_vix_m
469,2026-06-01,85.3991,"4,253.9864",3.9095,4.2100,4.4705,3.6300,17.9068
470,2026-07-01,83.7587,"4,088.7304",4.0509,4.3309,4.5995,3.6300,17.0909
471,2026-08-01,89.3271,"4,442.5200",4.0278,4.3678,4.6700,NaN,15.2390



Saved: /Users/darhanmashirapov/Desktop/MSF NU/Writing Sample MS2027/1. Data/29-35. Global:Common Factors /global_common_factors_merged.xlsx
